In [ ]:
!pip install easyocr pandas opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 32.8 MB/s eta 0:00:00


In [ ]:
import os
import re
import pandas as pd
import easyocr
import cv2
from google.colab import drive

In [ ]:
# Mount Google Drive to access your 10k images
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Initialize EasyOCR Reader (English)
reader = easyocr.Reader(['en'])

# Path to your images folder in Drive
IMAGE_FOLDER_PATH = '/content/drive/MyDrive/Raja Project/downloaded_images' # Update this!
CSV_FILE_PATH = '/content/drive/MyDrive/Raja Project/dataset2.csv' # Update this!
# CSV_FILE_PATH = '/content/drive/MyDrive/Raja Project/dataset1.csv'

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [ ]:
# Your units_dict from single_units.py
units_dict = {
    'width': {'c':'centimeter', 'cm': 'centimeter', 'm': 'meter', 'mm': 'millimeter', 'i':'inch', 'in': 'inch', 'f':'foot', 'ft': 'foot', 'y':'yard', 'yd': 'yard'},
    'depth': {'c':'centimeter', 'cm': 'centimeter', 'm': 'meter', 'mm': 'millimeter', 'i':'inch', 'in': 'inch', 'f':'foot', 'ft': 'foot', 'y':'yard', 'yd': 'yard'},
    'height': {'c':'centimeter', 'cm': 'centimeter', 'm': 'meter', 'mm': 'millimeter', 'i':'inch', 'in': 'inch', 'f':'foot', 'ft': 'foot', 'y':'yard', 'yd': 'yard'},
    'item_weight': {'g': 'gram', 'k':'kilogram', 'kg': 'kilogram', 'µ':'microgram', 'u':'microgram', 'µg': 'microgram', 'm':'milligram', 'mg': 'milligram', 'o':'ounce', 'oz': 'ounce', 'l':'pound', 'lb': 'pound', 't': 'ton'},
    'maximum_weight_recommendation': {'g': 'gram', 'k':'kilogram', 'kg': 'kilogram', 'µ':'microgram', 'u':'microgram', 'µg': 'microgram', 'm':'milligram', 'mg': 'milligram', 'o':'ounce', 'oz': 'ounce', 'l':'pound', 'lb': 'pound', 't': 'ton'},
    'voltage': {'k':'kilovolt', 'kv': 'kilovolt', 'm':'millivolt', 'mv': 'millivolt', 'v': 'volt'},
    'wattage': {'k':'kilowatt', 'kw': 'kilowatt', 'w': 'watt'},
    'item_volume': {'l': 'liter', 'm':'milliliter', 'ml': 'milliliter', 'c':'centiliter', 'cl': 'centiliter', 'd':'deciliter', 'dl': 'deciliter', 'u':'microliter', 'µ':'microliter', 'µl': 'microliter', 'f':'cubic foot', 'ft':'cubic foot', 'ft³': 'cubic foot', 'i':'cubic inch', 'in':'cubic inch', 'in³': 'cubic inch', 'cu':'cup', 'cup': 'cup', 'fl':'fluid ounce', 'flo':'fluid ounce', 'floz': 'fluid ounce', 'g':'gallon', 'ga':'gallon', 'gal': 'gallon', 'im':'imperial gallon', 'imp':'imperial gallon', 'impgal': 'imperial gallon', 'p':'pint', 'pt': 'pint', 'q':'quart', 'qt': 'quart'}
}

In [ ]:
def extract_value_with_unit(text, entity_name):
    if not text or entity_name not in units_dict:
        return ""

    # Get allowed units for this specific entity
    allowed_units = units_dict[entity_name]
    # Create regex pattern for units: looks for numbers followed by unit abbreviations
    # Matches: "120v", "120 v", "12.5 kg", etc.
    unit_patterns = sorted(allowed_units.keys(), key=len, reverse=True)
    pattern = r'(\d+(?:\.\d+)?)\s*(' + '|'.join(re.escape(u) for u in unit_patterns) + r')\b'

    match = re.search(pattern, text.lower())
    if match:
        number = match.group(1)
        unit_abbr = match.group(2)
        full_unit = allowed_units[unit_abbr]
        return f"{number} {full_unit}"
    return ""

def process_image(image_path, entity_name):
    try:
        if not os.path.exists(image_path):
            return "File Not Found", "", ""

        # OCR Extraction
        results = reader.readtext(image_path, detail=0)
        raw_text = " ".join(results)

        # Simple cleaning: remove non-alphanumeric except dots and spaces
        cleaned_text = re.sub(r'[^a-zA-Z0-9.\s]', '', raw_text)

        # Extract specific entity value
        entity_value = extract_value_with_unit(raw_text, entity_name)

        return raw_text, cleaned_text, entity_value
    except Exception as e:
        return f"Error: {str(e)}", "", ""

In [ ]:
# # Load the CSV
# df = pd.read_csv(CSV_FILE_PATH)

# # Initialize new columns
# df['raw_text'] = ""
# df['cleaned_text'] = ""
# df['entity_value'] = ""

# # Loop through the rows
# for index, row in df.iterrows():
#     # Extract filename from URL (e.g., 61I9XdN6OFL.jpg)
#     img_filename = row['image_link'].split('/')[-1]
#     img_full_path = os.path.join(IMAGE_FOLDER_PATH, img_filename)

#     print(f"Processing {index+1}/{len(df)}: {img_filename}")

#     raw, clean, val = process_image(img_full_path, row['entity_name'])

#     df.at[index, 'raw_text'] = raw
#     df.at[index, 'cleaned_text'] = clean
#     df.at[index, 'entity_value'] = val

# # Save the final output
# df.to_csv('final_output_with_extra_columns.csv', index=False)
# print("Processing Complete! File saved as final_output_with_extra_columns.csv")

In [ ]:
import os

# --- CONFIGURATION ---
BATCH_SIZE = 100  # Number of images to process before saving progress
OUTPUT_FILE = '/content/drive/MyDrive/final_output_batches.csv' # Path in Drive to save progress
# ---------------------

# Load the CSV
df = pd.read_csv(CSV_FILE_PATH)

# Initialize columns if they don't exist (important for resuming)
if 'raw_text' not in df.columns:
    df['raw_text'] = ""
if 'cleaned_text' not in df.columns:
    df['cleaned_text'] = ""
if 'entity_value' not in df.columns:
    df['entity_value'] = ""

# Check if a progress file already exists to resume
if os.path.exists(OUTPUT_FILE):
    print("Found existing progress file. Resuming...")
    df_progress = pd.read_csv(OUTPUT_FILE)
    # Update the main dataframe with already processed rows
    df.update(df_progress)
    # Find the first index where entity_value is still empty
    start_index = df[df['entity_value'] == ""].index[0] if not df[df['entity_value'] == ""].empty else len(df)
else:
    start_index = 0
    print("Starting from the beginning.")

total_rows = len(df)

# Loop through the rows in batches
for i in range(start_index, total_rows, BATCH_SIZE):
    end_index = min(i + BATCH_SIZE, total_rows)
    print(f"\n--- Processing Batch: {i} to {end_index} (Total: {total_rows}) ---")

    for index in range(i, end_index):
        row = df.iloc[index]

        # Extract filename from URL
        img_filename = row['image_link'].split('/')[-1]
        img_full_path = os.path.join(IMAGE_FOLDER_PATH, img_filename)

        # Process the image
        raw, clean, val = process_image(img_full_path, row['entity_name'])

        # Update DataFrame using .at for speed
        df.at[index, 'raw_text'] = raw
        df.at[index, 'cleaned_text'] = clean
        df.at[index, 'entity_value'] = val

        if (index + 1) % 10 == 0:
            print(f"  Processed {index + 1}/{total_rows}...")

    # Save progress to CSV after every batch
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"✓ Batch complete. Progress saved to {OUTPUT_FILE}")

print("\nProcessing finished successfully!")

Found existing progress file. Resuming...

--- Processing Batch: 1 to 101 (Total: 5000) ---
  Processed 10/5000...
  Processed 20/5000...
  Processed 30/5000...
  Processed 40/5000...
  Processed 50/5000...
  Processed 60/5000...
  Processed 70/5000...
  Processed 80/5000...
  Processed 90/5000...
  Processed 100/5000...
✓ Batch complete. Progress saved to /content/drive/MyDrive/final_output_batches.csv

--- Processing Batch: 101 to 201 (Total: 5000) ---
  Processed 110/5000...
  Processed 120/5000...
  Processed 130/5000...
  Processed 140/5000...
  Processed 150/5000...
  Processed 160/5000...
  Processed 170/5000...
  Processed 180/5000...
  Processed 190/5000...
  Processed 200/5000...
✓ Batch complete. Progress saved to /content/drive/MyDrive/final_output_batches.csv

--- Processing Batch: 201 to 301 (Total: 5000) ---
  Processed 210/5000...
  Processed 220/5000...
  Processed 230/5000...
  Processed 240/5000...
  Processed 250/5000...
  Processed 260/5000...
  Processed 270/5000.